# Lab Skills Sesi 2 — Aplikasi Coding AI untuk Radiologi Kedokteran Gigi

**Departemen Radiologi Dentomaksilofasial · PPDGS Radiologi Kedokteran Gigi · 150 menit**

> **Tujuan sesi ini adalah mengaudit artefak AI eksternal, bukan memakai AI sebagai alat layanan pasien.**
> Semua citra di notebook ini adalah kasus publik teranonimisasi. Jangan unggah radiograf pasien, nama,
> nomor rekam medis, tanggal lahir, atau metadata klinis ke Colab maupun layanan cloud lain.

Kita akan menjalankan detektor panoramik publik pada empat OPG pediatrik, mengubah ambang skor,
membandingkan kotak prediksi dengan anotasi yang kompatibel, lalu menghitung *occlusion sensitivity*
secara nyata. **Skor model bukan ukuran kepastian klinis; kotak dan heatmap bukan penjelasan kausal.**


## Hasil belajar

Setelah sesi ini, peserta mampu:

1. membedakan *classification*, *detection*, dan *segmentation*;
2. menjelaskan fungsi dataset card, model card, data train/test, dan label space;
3. menjalankan detektor ONNX pada CPU serta mengubah `CASE_ID` dan `CONF_THRESHOLD`;
4. membaca TP, FP, dan FN **hanya** untuk label yang dapat dipetakan;
5. membuat *occlusion sensitivity* dengan mengubah `OCCLUSION_GRID`; dan
6. menyebutkan keterbatasan domain shift, label mismatch, bias otomatisasi, privasi, dan kebutuhan pengawasan manusia.


## Agenda 150 menit

| Menit | Kegiatan |
|---:|---|
| 0–15 | Rekap, keamanan data, dan membaca kasus sebelum melihat keluaran AI |
| 15–45 | *Classification* vs *detection* vs *segmentation* |
| 45–75 | Dataset card, model card, train/test, dan label space |
| 75–90 | Istirahat + checkpoint |
| 90–115 | Inferensi empat kasus; bandingkan ambang 0,25 dan 0,45 |
| 115–135 | Anotasi vs prediksi, TP/FP/FN, label mismatch, dan domain shift |
| 135–150 | *Occlusion sensitivity*, etika, dan exit ticket |


## Cara memakai notebook

- Buka di **Google Colab**, pilih **Runtime → Change runtime type → CPU**, lalu jalankan sel dari atas ke bawah.
- `▶ Jalankan` berarti cukup tekan tombol play.
- `✏ Ubah` berarti hanya ubah satu parameter yang ditandai.
- `🩺 Diskusikan` berarti berhenti melihat kode dan hubungkan keluaran dengan penalaran radiologi.
- `✅ Checkpoint` berarti pastikan semua anggota kelompok dapat menjelaskan hasil sebelum lanjut.
- Kode panjang berada di sel **helper**. Boleh dibuka untuk dipelajari, tetapi tidak perlu diedit.

**Jika ada error:** baca baris terakhir → periksa apakah sel sebelumnya sudah dijalankan → jalankan ulang sel tersebut.
Jika status runtime hilang, pilih **Runtime → Restart session**, lalu **Run all**. Jika unduhan gagal,
cek koneksi dan jalankan kembali sel pemuatan. Notebook tidak akan memakai model lama yang checksum-nya salah.

**Preflight pengajar:** sehari sebelum kelas, jalankan *Run all* pada Colab CPU baru, pastikan checksum model cocok,
lalu simpan salinan repo beserta folder `assets/pediatric_opg`. Prediksi tersimpan tersedia untuk diskusi saat model
tidak dapat diunduh; prediksi itu selalu diberi label **FALLBACK**, bukan dipresentasikan sebagai inferensi baru.


---
## Menit 0–15 · Rekap dan baca kasus sebelum melihat AI

**Pertanyaan awal:** apa yang biasanya Anda periksa lebih dahulu pada OPG anak—kualitas citra, tahap perkembangan,
simetri, erupsi, atau temuan lokal? Tuliskan observasi citra dahulu. Jangan membuka anotasi atau keluaran model.


In [ ]:
# ▶ Jalankan — siapkan pustaka (CPU; tidak memerlukan GPU)
import importlib.metadata
import subprocess
import sys

REQUIRED_ORT = "1.27.0"
try:
    installed_ort = importlib.metadata.version("onnxruntime")
except importlib.metadata.PackageNotFoundError:
    installed_ort = None

if installed_ort != REQUIRED_ORT:
    print(f"Memasang onnxruntime=={REQUIRED_ORT}; perlu koneksi internet...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", f"onnxruntime=={REQUIRED_ORT}"]
    )

print("onnxruntime siap:", importlib.metadata.version("onnxruntime"))
print("Runtime: CPU")


In [ ]:
# ▶ Jalankan — helper aset publik (tidak perlu diedit)
import hashlib
import json
import tempfile
import urllib.error
import urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
from PIL import Image

ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/kristonova/"
    "introduction-AI-coding-for-radiologist/main/assets/pediatric_opg"
)
ASSET_CACHE = Path(tempfile.gettempdir()) / "dental_ai_lab_assets"
ASSET_CACHE.mkdir(parents=True, exist_ok=True)
REQUIRED_CASES = {
    "test_cate1_000", "test_cate1_001", "test_cate1_004", "test_cate1_012"
}

def _sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def _asset_candidates(filename):
    return [
        Path("assets") / "pediatric_opg" / filename,
        Path("..") / "assets" / "pediatric_opg" / filename,
        ASSET_CACHE / filename,
    ]

def _download_asset(filename, expected_sha256=None):
    destination = ASSET_CACHE / filename
    temporary = destination.with_suffix(destination.suffix + ".part")
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        if expected_sha256 and _sha256(destination) != expected_sha256:
            destination.unlink()
            print(f"Cache {filename} dihapus karena checksum tidak cocok.")
        else:
            return destination

    if temporary.exists():
        temporary.unlink()

    url = f"{ASSET_BASE_URL}/{filename}"
    request = urllib.request.Request(url, headers={"User-Agent": "Dental-AI-Lab/1.0"})
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            with open(temporary, "wb") as output:
                while True:
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    output.write(chunk)
    except (OSError, urllib.error.URLError) as error:
        if temporary.exists():
            temporary.unlink()
        raise RuntimeError(
            f"Gagal mengunduh {filename}. Periksa koneksi, lalu jalankan ulang sel. "
            "Jika kelas offline, letakkan folder assets/pediatric_opg di samping notebook."
        ) from error

    actual = _sha256(temporary)
    if expected_sha256 and actual != expected_sha256:
        temporary.unlink()
        raise RuntimeError(
            f"Checksum {filename} tidak cocok. File unduhan dihapus; "
            "notebook tidak akan memakai salinan lama."
        )
    temporary.replace(destination)
    return destination

def _find_asset(filename, expected_sha256=None):
    for candidate in _asset_candidates(filename):
        if not candidate.exists():
            continue
        if expected_sha256 and _sha256(candidate) != expected_sha256:
            if candidate == ASSET_CACHE / filename:
                candidate.unlink()
            raise RuntimeError(
                f"Checksum aset lokal {filename} tidak cocok. "
                "Ganti aset dengan salinan resmi sebelum melanjutkan."
            )
        return candidate
    return _download_asset(filename, expected_sha256)

def _manifest_cases(manifest):
    raw_cases = manifest.get("cases", manifest) if isinstance(manifest, dict) else manifest
    if isinstance(raw_cases, dict):
        cases = []
        for key, value in raw_cases.items():
            item = dict(value)
            item.setdefault("case_id", key)
            cases.append(item)
        return cases
    return list(raw_cases)

def load_manifest():
    manifest_path = _find_asset("cases.json")
    with open(manifest_path, encoding="utf-8") as handle:
        manifest = json.load(handle)

    index = {}
    for case in _manifest_cases(manifest):
        original_id = str(case["case_id"])
        prefixed_id = original_id if original_id.startswith("test_") else f"test_{original_id}"
        index[prefixed_id] = case
        index[original_id.removeprefix("test_")] = case

    missing = sorted(case_id for case_id in REQUIRED_CASES if case_id not in index)
    if missing:
        raise RuntimeError(f"Manifest tidak memuat kasus wajib: {missing}")
    return manifest, index

def _case_image_filename(case, requested_id):
    filename = case.get("image_file") or case.get("image") or case.get("filename")
    if filename:
        return str(filename)
    base_id = requested_id if requested_id.startswith("test_") else f"test_{requested_id}"
    return f"{base_id}.png"

def _case_annotations(case):
    annotations = case.get("annotations", case.get("labels", []))
    return list(annotations)

def load_case(case_id):
    manifest, index = load_manifest()
    lookup_id = case_id if case_id in index else case_id.removeprefix("test_")
    if lookup_id not in index:
        allowed = ", ".join(sorted(REQUIRED_CASES))
        raise ValueError(f"CASE_ID tidak dikenal. Pilih salah satu: {allowed}")

    case = index[lookup_id]
    filename = _case_image_filename(case, case_id)
    expected = case.get("image_sha256") or case.get("sha256")
    image_path = _find_asset(filename, expected)
    image_rgb = np.asarray(Image.open(image_path).convert("RGB"))
    if image_rgb.ndim != 3 or image_rgb.shape[2] != 3:
        raise RuntimeError("Citra gagal dikonversi menjadi RGB.")
    return image_rgb, case

def show_case_without_ai(image_rgb, case_id):
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(image_rgb)
    ax.set_title(f"{case_id} — baca citra sebelum melihat AI")
    ax.axis("off")
    plt.show()

print("Helper aset siap. Tidak ada data pasien yang diunggah.")


In [ ]:
# ✏ Ubah hanya CASE_ID; mulai dengan kasus deteksi parsial
CASE_ID = "test_cate1_004"


In [ ]:
# ▶ Jalankan — tampilkan OPG tanpa anotasi dan tanpa AI
if "load_case" not in globals() or "CASE_ID" not in globals():
    raise RuntimeError("Urutan sel belum siap. Pilih Runtime → Run all.")
image_rgb, case_info = load_case(CASE_ID)
show_case_without_ai(image_rgb, CASE_ID)
print("Ukuran citra (tinggi, lebar, kanal):", image_rgb.shape)


### 🩺 Diskusikan sebelum lanjut

Catat maksimal tiga observasi dan satu ketidakpastian. Bedakan **apa yang terlihat** dari **interpretasi**.
Setelah semua pasangan selesai, barulah buka bagian AI. Urutan ini membantu mengurangi *automation bias*:
kecenderungan mengubah penilaian hanya karena sudah melihat keluaran mesin.


---
## Menit 15–45 · *Classification*, *detection*, dan *segmentation*

| Tugas | Keluaran | Pertanyaan yang dijawab | Contoh batasan |
|---|---|---|---|
| Classification | satu/lebih label untuk citra | “Kelas apa ada pada citra?” | tidak menunjukkan lokasi |
| Detection | label + skor + kotak | “Apa dan kira-kira di mana?” | kotak tidak mengikuti bentuk lesi |
| Segmentation | label per piksel/mask | “Piksel mana termasuk objek?” | label rinci mahal dan tetap bisa keliru |

Detektor yang dipakai hari ini menghasilkan banyak kandidat kotak. `CONF_THRESHOLD` menyaring kandidat menurut
**skor model**, lalu NMS mengurangi kotak yang tumpang tindih. Angka itu bergantung pada data, model, dan pipeline;
jangan dibaca sebagai peluang penyakit pada seorang pasien.


### 🩺 Latihan konsep (3 menit)

Cocokkan kebutuhan berikut dengan tugas AI:

1. Menentukan apakah sebuah OPG perlu ditinjau ulang karena gerakan.
2. Memberi kotak pada kandidat lesi periapikal.
3. Menggambar kontur kanal mandibula piksel demi piksel.

<details>
<summary><b>Buka jawaban setelah berdiskusi</b></summary>

1. Classification (jika keluaran hanya label citra);
2. Detection;
3. Segmentation.

Nama tugas belum menjamin kegunaan klinis. Kualitas data, definisi label, dan validasi eksternal tetap harus diaudit.
</details>


### Label space: “kamus” kelas

Model hanya dapat mengeluarkan kelas yang ada pada label space-nya. Anotasi dataset hari ini mencakup karies,
periodontitis periapikal, pit/fisur dalam, pulpitis, kelainan perkembangan, dan lainnya. Model eksternal hanya
mengeluarkan `caries`, `periapical_lesion`, dan `impacted_tooth`.

Karena itu:

- karies ↔ `caries` dapat dibandingkan;
- periodontitis periapikal ↔ `periapical_lesion` dipakai sebagai pemetaan operasional yang harus dicatat;
- `impacted_tooth`, pit/fisur dalam, pulpitis, kelainan perkembangan, dan lainnya ditandai
  **“tidak dapat dinilai dengan skema label ini”**;
- kelas yang tidak kompatibel **tidak** boleh dipaksa menjadi TP, FP, atau FN.


---
## Menit 45–75 · Dataset card, model card, train/test, dan provenance

### Dataset card ringkas

Empat kasus berasal dari subset **Test** dataset publik
[Children’s Dental Panoramic Radiographs Dataset](https://springernature.figshare.com/articles/dataset/Children_s_Dental_Panoramic_Radiographs_Dataset/21621705).
Artikel pendamping [Scientific Data](https://www.nature.com/articles/s41597-023-02237-5)
menjelaskan 100 pasien pediatrik usia 2–13 tahun, proses anonimisasi, persetujuan, dan etik.
Record dataset menyatakan CC0; artikel menggunakan CC BY 4.0. Atribusi, DOI, dan SHA-256 setiap salinan
dicatat di manifest repo.

**Train** dipakai untuk membentuk parameter model; **validation** membantu memilih konfigurasi;
**test** disimpan untuk evaluasi akhir. Empat kasus ini tidak dipakai untuk melatih model di kelas dan
terlalu kecil untuk mengestimasi kinerja populasi.


In [ ]:
# ▶ Jalankan — lihat anotasi publik dan provenance kasus aktif
annotations = _case_annotations(case_info)
print("Kasus:", CASE_ID)
print("Split:", case_info.get("split", "Test"))
print("Jumlah anotasi:", len(annotations))
for number, annotation in enumerate(annotations, start=1):
    label_id = annotation.get("label_id") or annotation.get("label_indonesia")
    label_en = annotation.get("label_en") or annotation.get("label_english")
    print(f"{number:>2}. {label_id} / {label_en} | bbox={annotation.get('bbox_xyxy')}")


### Model card ringkas: artefak eksternal yang diaudit

Kita memakai `best.onnx` dari
[liodon-ai/dental-panoramic-detector](https://huggingface.co/liodon-ai/dental-panoramic-detector),
dipatok ke revisi `8bef2036b099e80e51f93f24de4b0c0edd366256`.

- SHA-256 yang wajib: `4cee38b54203634d895ed30a8910f5d7c4cefe22b18f9116b5561d9dd6e83a71`
- Input: RGB 640×640 dengan *letterbox* (rasio aspek dipertahankan)
- Inferensi: ONNX Runtime 1.27.0, CPU
- Rekomendasi model card: ambang skor 0,45; kelas membandingkan juga 0,25
- NMS IoU: 0,35
- Model card menyatakan CC BY-NC 4.0; metadata yang tertanam pada ONNX dapat menyebut lisensi Ultralytics
  AGPL-3.0. Perbedaan pemberitahuan lisensi harus diselesaikan sebelum penggunaan ulang/distribusi.

Model tidak didistribusikan bersama repo dan diunduh hanya untuk latihan akademik nonkomersial.
Provenance pelatihan serta representativitasnya terhadap OPG pediatrik ini harus diperlakukan sebagai
hal yang perlu diaudit, bukan diasumsikan.


In [ ]:
# ▶ Jalankan — helper model, inferensi, dan fallback (tidak perlu diedit)
import ast
import importlib.metadata

import onnxruntime as ort

MODEL_REVISION = "8bef2036b099e80e51f93f24de4b0c0edd366256"
MODEL_SHA256 = "4cee38b54203634d895ed30a8910f5d7c4cefe22b18f9116b5561d9dd6e83a71"
MODEL_URL = (
    "https://huggingface.co/liodon-ai/dental-panoramic-detector/resolve/"
    f"{MODEL_REVISION}/best.onnx?download=true"
)
MODEL_PATH = Path(tempfile.gettempdir()) / f"dental_detector_{MODEL_REVISION[:8]}.onnx"
DEFAULT_CLASS_NAMES = {0: "caries", 1: "periapical_lesion", 2: "impacted_tooth"}
BOX_COLORS = {
    "caries": "#ff3b30",
    "periapical_lesion": "#ff9500",
    "impacted_tooth": "#00a6fb",
}

def _download_verified_model():
    temporary = MODEL_PATH.with_suffix(".onnx.part")
    if MODEL_PATH.exists():
        actual = _sha256(MODEL_PATH)
        if actual == MODEL_SHA256:
            print("Model terverifikasi dari cache:", actual[:12] + "…")
            return MODEL_PATH
        MODEL_PATH.unlink()
        print("Model cache dihapus karena checksum tidak cocok.")
    if temporary.exists():
        temporary.unlink()

    print("Mengunduh model ONNX terpatok (sekitar 11 MB)...")
    request = urllib.request.Request(MODEL_URL, headers={"User-Agent": "Dental-AI-Lab/1.0"})
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            with open(temporary, "wb") as output:
                while True:
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    output.write(chunk)
    except (OSError, urllib.error.URLError) as error:
        if temporary.exists():
            temporary.unlink()
        raise RuntimeError(
            "Unduhan model gagal atau timeout. File parsial dihapus. "
            "Periksa koneksi, lalu jalankan ulang sel ini."
        ) from error

    actual = _sha256(temporary)
    if actual != MODEL_SHA256:
        temporary.unlink()
        raise RuntimeError(
            "Checksum model tidak cocok. File unduhan dihapus dan tidak dimuat. "
            f"Diperoleh {actual}; diharapkan {MODEL_SHA256}."
        )
    temporary.replace(MODEL_PATH)
    print("Checksum model cocok:", actual)
    return MODEL_PATH

def _parse_class_names(metadata):
    raw = metadata.get("names") if metadata else None
    if not raw:
        return DEFAULT_CLASS_NAMES.copy()
    try:
        parsed = ast.literal_eval(raw)
        return {int(key): str(value) for key, value in parsed.items()}
    except (ValueError, SyntaxError, AttributeError):
        print("Nama kelas metadata tidak terbaca; memakai nama dari model card.")
        return DEFAULT_CLASS_NAMES.copy()

def load_detector():
    if importlib.metadata.version("onnxruntime") != "1.27.0":
        raise RuntimeError(
            "Versi onnxruntime bukan 1.27.0. Jalankan kembali sel persiapan, "
            "restart session bila diminta, lalu ulangi."
        )
    model_path = _download_verified_model()
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1
    options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session = ort.InferenceSession(
        str(model_path), sess_options=options, providers=["CPUExecutionProvider"]
    )
    metadata = session.get_modelmeta().custom_metadata_map
    detector = {
        "session": session,
        "input_name": session.get_inputs()[0].name,
        "names": _parse_class_names(metadata),
        "metadata": metadata,
        "sha256": _sha256(model_path),
    }
    if detector["sha256"] != MODEL_SHA256:
        raise RuntimeError("Checksum berubah setelah model dimuat; inferensi dihentikan.")
    return detector

def _letterbox(image_rgb, size=640):
    height, width = image_rgb.shape[:2]
    scale = min(size / width, size / height)
    resized_width = int(round(width * scale))
    resized_height = int(round(height * scale))
    resized = cv2.resize(
        image_rgb, (resized_width, resized_height), interpolation=cv2.INTER_LINEAR
    )
    pad_x = size - resized_width
    pad_y = size - resized_height
    left = pad_x // 2
    top = pad_y // 2
    canvas = np.full((size, size, 3), 114, dtype=np.uint8)
    canvas[top:top + resized_height, left:left + resized_width] = resized
    geometry = {
        "scale": scale,
        "left": left,
        "top": top,
        "resized_width": resized_width,
        "resized_height": resized_height,
        "original_width": width,
        "original_height": height,
        "size": size,
    }
    tensor = canvas.astype(np.float32) / 255.0
    tensor = np.transpose(tensor, (2, 0, 1))[None, ...]
    return tensor, geometry

def _output_rows(detector, tensor):
    raw = detector["session"].run(
        None, {detector["input_name"]: tensor.astype(np.float32)}
    )[0]
    rows = np.squeeze(raw)
    if rows.ndim != 2:
        raise RuntimeError(f"Shape output model tidak dikenali: {raw.shape}")
    if rows.shape[0] <= 100 and rows.shape[1] > rows.shape[0]:
        rows = rows.T
    if rows.shape[1] < 5:
        raise RuntimeError(f"Output tidak memuat skor kelas: {rows.shape}")
    return rows

def _xywh_to_original(xywh, geometry):
    center_x, center_y, width, height = [float(value) for value in xywh]
    x1 = (center_x - width / 2 - geometry["left"]) / geometry["scale"]
    y1 = (center_y - height / 2 - geometry["top"]) / geometry["scale"]
    x2 = (center_x + width / 2 - geometry["left"]) / geometry["scale"]
    y2 = (center_y + height / 2 - geometry["top"]) / geometry["scale"]
    x1 = np.clip(x1, 0, geometry["original_width"])
    x2 = np.clip(x2, 0, geometry["original_width"])
    y1 = np.clip(y1, 0, geometry["original_height"])
    y2 = np.clip(y2, 0, geometry["original_height"])
    return [float(x1), float(y1), float(x2), float(y2)]

def _box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = [float(value) for value in box_a]
    bx1, by1, bx2, by2 = [float(value) for value in box_b]
    intersection_width = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    intersection_height = max(0.0, min(ay2, by2) - max(ay1, by1))
    intersection = intersection_width * intersection_height
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0

def _classwise_nms(candidates, iou_threshold):
    kept = []
    class_ids = sorted({candidate["class_id"] for candidate in candidates})
    for class_id in class_ids:
        remaining = sorted(
            [candidate for candidate in candidates if candidate["class_id"] == class_id],
            key=lambda candidate: candidate["score"],
            reverse=True,
        )
        while remaining:
            best = remaining.pop(0)
            kept.append(best)
            remaining = [
                candidate for candidate in remaining
                if _box_iou(best["bbox_xyxy"], candidate["bbox_xyxy"]) <= iou_threshold
            ]
    return sorted(kept, key=lambda candidate: candidate["score"], reverse=True)

def predict_boxes(detector, image_rgb, conf_threshold=0.25, nms_iou=0.35):
    if detector is None:
        raise RuntimeError("Detektor belum tersedia; muat model terlebih dahulu.")
    if not 0.0 <= conf_threshold <= 1.0:
        raise ValueError("conf_threshold harus berada di antara 0 dan 1.")
    tensor, geometry = _letterbox(image_rgb, size=640)
    rows = _output_rows(detector, tensor)
    class_scores = rows[:, 4:]
    class_ids = np.argmax(class_scores, axis=1)
    scores = class_scores[np.arange(len(rows)), class_ids]
    candidates = []
    for row, class_id, score in zip(rows, class_ids, scores):
        if float(score) < conf_threshold:
            continue
        box = _xywh_to_original(row[:4], geometry)
        if box[2] <= box[0] or box[3] <= box[1]:
            continue
        candidates.append(
            {
                "class_id": int(class_id),
                "label": detector["names"].get(int(class_id), f"class_{class_id}"),
                "score": float(score),
                "bbox_xyxy": box,
            }
        )
    return _classwise_nms(candidates, nms_iou)

def _normalize_prediction(prediction):
    result = dict(prediction)
    result["bbox_xyxy"] = [
        float(value) for value in result.get("bbox_xyxy", result.get("box", []))
    ]
    result["score"] = float(result["score"])
    result["label"] = str(result["label"])
    if "class_id" in result:
        result["class_id"] = int(result["class_id"])
    else:
        reverse_names = {value: key for key, value in DEFAULT_CLASS_NAMES.items()}
        result["class_id"] = reverse_names.get(result["label"], -1)
    return result

def load_fallback_predictions():
    path = _find_asset("precomputed_predictions.json")
    with open(path, encoding="utf-8") as handle:
        data = json.load(handle)
    recorded_hash = data.get("model_sha256")
    if not recorded_hash and isinstance(data.get("model"), dict):
        recorded_hash = data["model"].get("sha256")
    if recorded_hash and recorded_hash != MODEL_SHA256:
        raise RuntimeError("Fallback berasal dari hash model yang berbeda; fallback ditolak.")
    return data

def _fallback_for(data, case_id, threshold):
    predictions_root = data.get("predictions", data.get("cases", data))
    keys = [case_id, case_id.removeprefix("test_"), f"test_{case_id.removeprefix('test_')}"]
    case_block = next((predictions_root[key] for key in keys if key in predictions_root), None)
    if case_block is None:
        raise KeyError(f"Fallback untuk {case_id} tidak tersedia.")
    threshold_keys = [f"{threshold:.2f}", str(threshold), str(round(threshold, 2))]
    result = next((case_block[key] for key in threshold_keys if key in case_block), None)
    if result is None:
        available = ", ".join(sorted(case_block))
        raise KeyError(
            f"Fallback hanya tersedia untuk ambang: {available}. "
            "Gunakan inferensi langsung untuk ambang lain."
        )
    return [_normalize_prediction(item) for item in result]

def show_predictions(image, predictions, title, ax=None):
    created = ax is None
    if created:
        _, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(image)
    for prediction in predictions:
        x1, y1, x2, y2 = prediction["bbox_xyxy"]
        color = BOX_COLORS.get(prediction["label"], "#7b61ff")
        ax.add_patch(
            Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=2, ec=color)
        )
        ax.text(
            x1,
            max(0, y1 - 7),
            f"{prediction['label']} | skor {prediction['score']:.2f}",
            color="white",
            fontsize=8,
            bbox={"facecolor": color, "alpha": 0.85, "pad": 2, "edgecolor": "none"},
        )
    ax.set_title(title)
    ax.axis("off")
    if created:
        plt.show()

def get_case_predictions(case_id, threshold):
    image, case = load_case(case_id)
    detector_value = globals().get("DETECTOR")
    if detector_value is not None:
        predictions = predict_boxes(
            detector_value, image, conf_threshold=threshold, nms_iou=0.35
        )
        source = "INFERENSI LANGSUNG · ONNX terverifikasi"
    else:
        fallback_data = globals().get("FALLBACK_DATA")
        if fallback_data is None:
            raise RuntimeError(
                "Model dan fallback belum tersedia. Jalankan sel pemuatan model."
            )
        predictions = _fallback_for(fallback_data, case_id, threshold)
        source = "FALLBACK TERSIMPAN · hanya untuk diskusi"
    return image, case, predictions, source

print("Helper detektor siap. NMS IoU dikunci pada 0,35.")


In [ ]:
# ▶ Jalankan — unduh, verifikasi, dan muat model pada CPU
if "load_detector" not in globals() or "load_fallback_predictions" not in globals():
    raise RuntimeError("Helper model belum siap. Pilih Runtime → Run all.")
DETECTOR = None
FALLBACK_DATA = None
try:
    DETECTOR = load_detector()
    print("Mode: INFERENSI LANGSUNG")
    print("Kelas model:", DETECTOR["names"])
    print("Provider:", DETECTOR["session"].get_providers())
    print("Metadata license:", DETECTOR["metadata"].get("license", "tidak tercatat"))
except Exception as error:
    print("MODEL TIDAK DIMUAT:", error)
    print("Beralih ke FALLBACK TERSIMPAN hanya untuk latihan membaca overlay.")
    FALLBACK_DATA = load_fallback_predictions()


> **Pemulihan model:** bila muncul `MODEL TIDAK DIMUAT`, baca penyebab tepat di atas. Untuk timeout,
> periksa koneksi lalu jalankan ulang sel pemuatan. Untuk checksum salah, jangan mengganti nilai checksum;
> unduh ulang dari revisi terpatok. Fallback hanya mempertahankan kegiatan diskusi overlay dan **tidak**
> dapat membuat heatmap baru.

### ✅ Checkpoint + istirahat (menit 75–90)

Sebelum istirahat, setiap pasangan harus dapat menjawab: (1) apa label space model, (2) mengapa empat kasus test
bukan bukti kinerja populasi, dan (3) apa beda skor model dengan kepastian klinis.


---
## Menit 90–115 · Inferensi empat kasus dan perubahan threshold

`CONF_THRESHOLD` adalah ambang skor minimal sebelum NMS. Menurunkannya biasanya menambah kandidat—termasuk
kandidat keliru. Menaikkannya biasanya mengurangi kandidat—termasuk kandidat yang mungkin relevan.


In [ ]:
# ✏ Ubah satu angka: coba 0.25 lalu 0.45
CONF_THRESHOLD = 0.25


In [ ]:
# ▶ Jalankan — inferensi kasus aktif
if not all(name in globals() for name in ("get_case_predictions", "CASE_ID", "CONF_THRESHOLD")):
    raise RuntimeError("Urutan sel belum lengkap. Pilih Runtime → Run all.")
image_rgb, case_info, predictions, prediction_source = get_case_predictions(
    CASE_ID, CONF_THRESHOLD
)
show_predictions(
    image_rgb,
    predictions,
    f"{CASE_ID} · conf={CONF_THRESHOLD:.2f}\n{prediction_source}",
)
print("Jumlah kotak setelah NMS:", len(predictions))
print("Sumber keluaran:", prediction_source)


In [ ]:
# ▶ Jalankan — bandingkan 0,25 dengan rekomendasi model card 0,45
comparison = []
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, threshold in zip(axes, [0.25, 0.45]):
    image, _, boxes, source = get_case_predictions(CASE_ID, threshold)
    comparison.append((threshold, boxes))
    show_predictions(
        image, boxes, f"conf={threshold:.2f} · {len(boxes)} kotak\n{source}", ax=ax
    )
plt.tight_layout()
plt.show()


In [ ]:
# ▶ Jalankan — audit visual empat kasus publik pada conf=0,25
AUDIT_CASES = [
    "test_cate1_004",  # deteksi parsial
    "test_cate1_012",  # karies + periapikal
    "test_cate1_001",  # sebagian label di luar cakupan model
    "test_cate1_000",  # label-space mismatch
]
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
for ax, audit_case in zip(axes.ravel(), AUDIT_CASES):
    image, _, boxes, source = get_case_predictions(audit_case, 0.25)
    show_predictions(image, boxes, f"{audit_case} · {len(boxes)} kotak\n{source}", ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
# ▶ Jalankan — galeri kanonis hanya saat mode FALLBACK
if DETECTOR is None:
    fig, axes = plt.subplots(2, 4, figsize=(20, 9))
    for row, (threshold_key, threshold_label) in enumerate(
        [("conf025", "conf=0,25"), ("conf045", "conf=0,45")]
    ):
        for column, audit_case in enumerate(AUDIT_CASES):
            filename = (
                f"canonical_overlay_{audit_case}_{threshold_key}.png"
            )
            canonical_path = _find_asset(filename)
            canonical_rgb = np.asarray(
                Image.open(canonical_path).convert("RGB")
            )
            axes[row, column].imshow(canonical_rgb)
            axes[row, column].set_title(
                f"{audit_case} · {threshold_label}\n"
                "OUTPUT TERSIMPAN"
            )
            axes[row, column].axis("off")
    fig.suptitle("BUKAN INFERENSI SESI INI", color="#b00020", fontsize=15)
    plt.tight_layout()
    plt.show()
else:
    print("Inferensi langsung tersedia; galeri output tersimpan tidak diperlukan.")


### 🩺 Diskusikan

- Kotak apa yang hilang saat ambang naik dari 0,25 ke 0,45?
- Apakah kotak dengan skor tertinggi selalu paling masuk akal secara anatomi?
- Pada `test_cate1_000`, anotasi publik hanya kelas “lainnya”. Apakah kemunculan kotak model dapat langsung
  disebut salah? Tidak—label referensi tidak cukup untuk menilai semua kelas model.
- Pada `test_cate1_012`, perhatikan apakah model menemukan kedua jenis label. Tidak terdeteksi bukan berarti
  lesi tidak ada; itu adalah kandidat **false negative** hanya jika label dan kriteria referensinya kompatibel.


---
## Menit 115–135 · Anotasi vs prediksi, TP/FP/FN, dan domain shift

Untuk latihan ini, kotak dipasangkan bila kelas sama dan IoU ≥ 0,50:

- **TP:** satu prediksi kompatibel berhasil dipasangkan dengan satu anotasi;
- **FP:** prediksi kelas kompatibel tidak mendapat pasangan;
- **FN:** anotasi kelas kompatibel tidak mendapat pasangan.

Angka ini bergantung pada definisi label, IoU, dan kualitas anotasi. Ia bukan ukuran keberhasilan klinis.


In [ ]:
# ▶ Jalankan — helper pencocokan label yang kompatibel (tidak perlu diedit)
SUPPORTED_LABELS = {"caries", "periapical_lesion"}

def _annotation_name(annotation):
    values = [
        annotation.get("label_en"),
        annotation.get("label_english"),
        annotation.get("label_id"),
        annotation.get("label_indonesia"),
        annotation.get("label"),
    ]
    return " | ".join(str(value) for value in values if value).lower()

def _mapped_annotation_label(annotation):
    name = _annotation_name(annotation)
    if "caries" in name or "karies" in name or "龋病" in name:
        return "caries"
    if (
        "periapical" in name
        or "periapikal" in name
        or "apical periodontitis" in name
        or "根尖周炎" in name
    ):
        return "periapical_lesion"
    return None

def match_predictions(predictions, annotations, iou_threshold=0.50):
    supported_ground_truth = []
    unscorable_ground_truth = []
    for index, annotation in enumerate(annotations):
        mapped = _mapped_annotation_label(annotation)
        item = {
            "index": index,
            "label": mapped,
            "original_label": _annotation_name(annotation),
            "bbox_xyxy": [float(value) for value in annotation["bbox_xyxy"]],
        }
        if mapped is None:
            unscorable_ground_truth.append(item)
        else:
            supported_ground_truth.append(item)

    supported_predictions = []
    unscorable_predictions = []
    for index, prediction in enumerate(predictions):
        item = dict(prediction)
        item["index"] = index
        if prediction["label"] in SUPPORTED_LABELS:
            supported_predictions.append(item)
        else:
            unscorable_predictions.append(item)

    candidate_pairs = []
    for prediction in supported_predictions:
        for ground_truth in supported_ground_truth:
            if prediction["label"] != ground_truth["label"]:
                continue
            iou = _box_iou(prediction["bbox_xyxy"], ground_truth["bbox_xyxy"])
            if iou >= iou_threshold:
                candidate_pairs.append((iou, prediction, ground_truth))
    candidate_pairs.sort(key=lambda item: item[0], reverse=True)

    matched_predictions = set()
    matched_ground_truth = set()
    matches = []
    for iou, prediction, ground_truth in candidate_pairs:
        if prediction["index"] in matched_predictions:
            continue
        if ground_truth["index"] in matched_ground_truth:
            continue
        matched_predictions.add(prediction["index"])
        matched_ground_truth.add(ground_truth["index"])
        matches.append(
            {
                "prediction_index": prediction["index"],
                "ground_truth_index": ground_truth["index"],
                "label": prediction["label"],
                "iou": float(iou),
            }
        )

    false_positives = [
        item for item in supported_predictions if item["index"] not in matched_predictions
    ]
    false_negatives = [
        item for item in supported_ground_truth if item["index"] not in matched_ground_truth
    ]
    return {
        "tp": len(matches),
        "fp": len(false_positives),
        "fn": len(false_negatives),
        "matches": matches,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "unscorable_ground_truth": unscorable_ground_truth,
        "unscorable_predictions": unscorable_predictions,
    }

def print_match_summary(case_id, threshold, result):
    print(
        f"{case_id} | conf={threshold:.2f} | "
        f"TP={result['tp']} FP={result['fp']} FN={result['fn']}"
    )
    for item in result["unscorable_ground_truth"]:
        print("  GT:", item["original_label"], "=> tidak dapat dinilai dengan skema label ini")
    for item in result["unscorable_predictions"]:
        print(
            "  Prediksi:", item["label"],
            "=> tidak dapat dinilai dengan skema label ini"
        )

print("Helper evaluasi siap. IoU pencocokan dikunci pada 0,50.")


In [ ]:
# ▶ Jalankan — hitung TP/FP/FN kasus aktif, hanya label kompatibel
evaluation = match_predictions(
    predictions, _case_annotations(case_info), iou_threshold=0.50
)
print_match_summary(CASE_ID, CONF_THRESHOLD, evaluation)
print("Jumlah pasangan dan IoU:", evaluation["matches"])


In [ ]:
# ▶ Jalankan — ringkasan empat kasus pada conf=0,25
for audit_case in AUDIT_CASES:
    _, audit_info, audit_boxes, _ = get_case_predictions(audit_case, 0.25)
    result = match_predictions(
        audit_boxes, _case_annotations(audit_info), iou_threshold=0.50
    )
    print_match_summary(audit_case, 0.25, result)
    print()


### Mengapa performa dapat berubah?

- **Domain shift:** usia, tahap dentisi, alat, protokol, posisi, resolusi, dan distribusi penyakit dapat berbeda
  dari data yang membentuk model. Kegagalan pada empat citra konsisten dengan kemungkinan domain shift,
  tetapi sampel ini tidak cukup untuk menetapkan penyebab tunggal.
- **Class imbalance:** kelas jarang memberi lebih sedikit contoh untuk dipelajari dan interval ketidakpastian
  lebih lebar. Empat kasus ini tidak boleh dipakai menghitung klaim sensitivitas/spesifisitas.
- **Label mismatch:** “periodontitis periapikal” dipetakan secara operasional ke `periapical_lesion`;
  definisi anotator dan model belum tentu identik.
- **False positive / false negative:** dampaknya tidak simetris dan bergantung pada alur klinis.
  Ambang seharusnya dipilih melalui validasi pada populasi tujuan, bukan karena satu gambar terlihat lebih baik.

### ✅ Checkpoint

<details>
<summary><b>Self-check: buka setelah menjawab</b></summary>

**Mengapa `impacted_tooth` tidak dihitung FP pada latihan ini?** Karena anotasi referensi yang dipakai tidak
menyediakan skema kompatibel untuk menilai kelas tersebut. Ketiadaan anotasi bukan bukti ketiadaan objek.

**Apakah menaikkan threshold selalu memperbaiki model?** Tidak. Ia mengubah trade-off kandidat berlebih dan
kandidat yang hilang; pilihan perlu divalidasi pada konteks tujuan.
</details>


---
## Menit 135–150 · *Occlusion sensitivity*, etika, dan exit ticket

Metode ini menutup satu bagian input 640×640 bergantian dengan nilai abu-abu letterbox, lalu mengukur perubahan
skor maksimum kelas `caries`. Grid 6×6 berarti 36 inferensi tambahan. Nilai positif menunjukkan skor turun
saat area ditutup; nilai negatif berarti skor justru naik.


In [ ]:
# ✏ Ubah satu angka: bandingkan 6 dengan 4 atau 8
OCCLUSION_GRID = 6


In [ ]:
# ▶ Jalankan — helper occlusion sensitivity nyata (tidak perlu diedit)
def _target_class_id(detector, target_class):
    reverse_names = {name: class_id for class_id, name in detector["names"].items()}
    if target_class not in reverse_names:
        raise ValueError(f"Kelas {target_class!r} tidak ada pada model.")
    return reverse_names[target_class]

def _maximum_raw_class_score(detector, tensor, target_class_id):
    rows = _output_rows(detector, tensor)
    score_column = 4 + int(target_class_id)
    if score_column >= rows.shape[1]:
        raise RuntimeError("Indeks kelas berada di luar output model.")
    return float(np.max(rows[:, score_column]))

def occlusion_sensitivity(detector, image_rgb, target_class="caries", grid_size=6):
    if detector is None:
        raise RuntimeError(
            "Occlusion sensitivity memerlukan inferensi langsung. "
            "Fallback tidak boleh digunakan untuk membuat heatmap."
        )
    if not isinstance(grid_size, int) or not 2 <= grid_size <= 12:
        raise ValueError("grid_size harus bilangan bulat 2–12.")

    tensor, geometry = _letterbox(image_rgb, size=640)
    target_class_id = _target_class_id(detector, target_class)
    baseline_score = _maximum_raw_class_score(detector, tensor, target_class_id)
    heatmap_grid = np.zeros((grid_size, grid_size), dtype=np.float32)
    edges = np.linspace(0, geometry["size"], grid_size + 1, dtype=int)

    for row in range(grid_size):
        for column in range(grid_size):
            occluded = tensor.copy()
            y1, y2 = edges[row], edges[row + 1]
            x1, x2 = edges[column], edges[column + 1]
            occluded[:, :, y1:y2, x1:x2] = 114.0 / 255.0
            occluded_score = _maximum_raw_class_score(
                detector, occluded, target_class_id
            )
            heatmap_grid[row, column] = baseline_score - occluded_score
        print(f"Baris grid {row + 1}/{grid_size} selesai")

    heatmap_letterbox = cv2.resize(
        heatmap_grid,
        (geometry["size"], geometry["size"]),
        interpolation=cv2.INTER_NEAREST,
    )
    top = geometry["top"]
    left = geometry["left"]
    cropped = heatmap_letterbox[
        top:top + geometry["resized_height"],
        left:left + geometry["resized_width"],
    ]
    heatmap_original = cv2.resize(
        cropped,
        (geometry["original_width"], geometry["original_height"]),
        interpolation=cv2.INTER_LINEAR,
    )
    if not np.isfinite(heatmap_original).all():
        raise RuntimeError("Heatmap memuat nilai non-finite.")
    return {
        "target_class": target_class,
        "baseline_score": baseline_score,
        "heatmap_grid": heatmap_grid,
        "heatmap_original": heatmap_original,
    }

def show_occlusion(image, result, annotations):
    positive_drop = np.maximum(result["heatmap_original"], 0)
    maximum = float(positive_drop.max())
    normalized = positive_drop / maximum if maximum > 0 else positive_drop

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    axes[0].imshow(image)
    axes[0].set_title("Citra + anotasi karies yang kompatibel")
    for annotation in annotations:
        if _mapped_annotation_label(annotation) != "caries":
            continue
        x1, y1, x2, y2 = annotation["bbox_xyxy"]
        axes[0].add_patch(
            Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="#34c759", lw=2)
        )
    axes[0].axis("off")

    axes[1].imshow(image)
    overlay = axes[1].imshow(
        normalized, cmap="inferno", alpha=0.50, vmin=0, vmax=1
    )
    axes[1].set_title(
        f"Occlusion sensitivity · {result['target_class']}\n"
        f"skor dasar mentah={result['baseline_score']:.3f}"
    )
    axes[1].axis("off")
    fig.colorbar(overlay, ax=axes[1], fraction=0.03, label="penurunan skor positif (dinormalisasi)")
    plt.tight_layout()
    plt.show()
    print(
        "Rentang perubahan skor (dasar − setelah ditutup):",
        f"{result['heatmap_grid'].min():.4f} s.d. {result['heatmap_grid'].max():.4f}",
    )

print("Helper occlusion siap.")


In [ ]:
# ▶ Jalankan — 36 inferensi tambahan bila grid=6
if not all(name in globals() for name in ("DETECTOR", "image_rgb", "OCCLUSION_GRID")):
    raise RuntimeError("Urutan sel belum lengkap. Pilih Runtime → Run all.")
OCCLUSION_RESULT = None
if DETECTOR is None:
    print(
        "Occlusion tidak dijalankan karena model langsung tidak tersedia. "
        "Tidak dibuat heatmap baru dari fallback."
    )
    canonical_path = _find_asset(
        "canonical_occlusion_test_cate1_004_grid6.png"
    )
    canonical_rgb = np.asarray(Image.open(canonical_path).convert("RGB"))
    plt.figure(figsize=(14, 6))
    plt.imshow(canonical_rgb)
    plt.title(
        "test_cate1_004 · OUTPUT TERSIMPAN GRID 6\n"
        "BUKAN PERHITUNGAN SESI INI"
    )
    plt.axis("off")
    plt.show()
else:
    OCCLUSION_RESULT = occlusion_sensitivity(
        DETECTOR, image_rgb, target_class="caries", grid_size=OCCLUSION_GRID
    )
    show_occlusion(
        image_rgb, OCCLUSION_RESULT, _case_annotations(case_info)
    )


### 🩺 Membaca heatmap dengan aman

- Heatmap hanya menunjukkan **sensitivitas skor terhadap pola penutupan yang kita pilih**.
- Grid yang lebih kasar menutup area lebih luas; grid lebih rapat meningkatkan resolusi tetapi juga jumlah inferensi
  dan dapat menghasilkan pola berbeda.
- Area panas di luar anotasi bisa berasal dari konteks, artefak penutupan, atau fitur korelatif.
- Area yang berimpit dengan kotak anotasi belum membuktikan model memakai tanda radiografis yang benar.
- Kotak prediksi bukan penjelasan kausal; heatmap bukan bukti lesi dan bukan validasi model.

Bandingkan grid 4, 6, dan 8. Jika kesimpulan berubah hanya karena grid berubah, penjelasan tersebut tidak stabil.


In [ ]:
# ✅ Checkpoint teknis — overlay, koordinat, heatmap, dan label unsupported
assert image_rgb.dtype == np.uint8
assert image_rgb.ndim == 3 and image_rgb.shape[2] == 3
height, width = image_rgb.shape[:2]
for prediction in predictions:
    x1, y1, x2, y2 = prediction["bbox_xyxy"]
    assert 0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height
unsupported_test = match_predictions(
    [{"label": "impacted_tooth", "score": 0.9, "bbox_xyxy": [0, 0, 10, 10]}],
    [],
)
assert unsupported_test["fp"] == 0
assert len(unsupported_test["unscorable_predictions"]) == 1
if OCCLUSION_RESULT is not None:
    heatmap = OCCLUSION_RESULT["heatmap_original"]
    assert heatmap.shape == image_rgb.shape[:2]
    assert np.isfinite(heatmap).all()
print("Checkpoint teknis lulus.")


## Penggunaan bertanggung jawab dan human oversight

- **Automation bias:** baca citra dahulu; tampilkan AI setelah observasi awal dicatat.
- **Privasi cloud:** jangan unggah data pasien tanpa dasar hukum, persetujuan, pengaturan institusi, dan kontrol vendor
  yang sesuai. Untuk kelas ini hanya case ID publik yang boleh dipakai.
- **Human oversight:** operator bertanggung jawab memeriksa kualitas citra, konteks, label yang tidak dicakup,
  serta konsekuensi FP/FN. AI tidak menggantikan kompetensi radiolog.
- **Validasi lokal:** sebelum penggunaan nyata diperlukan protokol, populasi target, pembanding, metrik, analisis
  subkelompok, kalibrasi, keamanan, pemantauan drift, dan jalur eskalasi.
- **Lisensi dan provenance:** audit hak penggunaan data/model serta semua dependensi sebelum redistribusi.

**Aturan praktis:** jika Anda tidak dapat menjelaskan asal data, label space, versi model, preprocessing,
threshold, dan keterbatasan evaluasi, keluaran belum layak dijadikan dasar tindakan.


## Exit ticket — tulis empat jawaban singkat

1. Apa perubahan yang Anda lihat ketika `CONF_THRESHOLD` diubah dari 0,25 menjadi 0,45?
2. Sebutkan satu label yang tidak dapat dinilai dan jelaskan mengapa tidak boleh dimasukkan ke FP/FN.
3. Apa arti satu area panas pada *occlusion sensitivity*, dan apa yang **tidak** dapat disimpulkan darinya?
4. Sebutkan satu risiko penerapan pada OPG pediatrik dan satu kontrol manusia yang diperlukan.

**Kriteria selesai:** jawaban menyebut data/model/threshold secara spesifik, bukan hanya “AI dapat salah”.


## Glossary

| Istilah | Arti praktis |
|---|---|
| Bounding box | Kotak `[x1, y1, x2, y2]` yang memperkirakan lokasi objek |
| Confidence threshold | Ambang skor model untuk mempertahankan kandidat |
| NMS | Prosedur mengurangi kotak tumpang tindih; di sini IoU 0,35 |
| IoU | Luas irisan dibagi luas gabungan dua kotak |
| TP / FP / FN | Ringkasan pasangan prediksi–anotasi dalam definisi evaluasi tertentu |
| Label space | Daftar kelas dan definisinya |
| Domain shift | Perbedaan distribusi data pengembangan dan data tujuan |
| Occlusion sensitivity | Perubahan skor ketika sebagian input ditutup |
| Fallback | Prediksi tersimpan untuk diskusi; bukan inferensi pada sesi saat ini |
| Human oversight | Pengawasan manusia atas input, keluaran, dampak, dan eskalasi |


## Sumber dan batas penggunaan

- Dataset publik: [Children’s Dental Panoramic Radiographs Dataset (Figshare)](https://springernature.figshare.com/articles/dataset/Children_s_Dental_Panoramic_Radiographs_Dataset/21621705)
- Data descriptor: [Scientific Data, DOI 10.1038/s41597-023-02237-5](https://doi.org/10.1038/s41597-023-02237-5)
- Model eksternal: [liodon-ai/dental-panoramic-detector](https://huggingface.co/liodon-ai/dental-panoramic-detector)
- Runtime: [ONNX Runtime](https://onnxruntime.ai/)

Materi ini untuk pendidikan dan audit teknis nonklinis. Empat kasus publik tidak mewakili seluruh populasi,
keluaran tidak boleh dipakai untuk keputusan perawatan, dan notebook memerlukan koneksi untuk unduhan awal
aset/model kecuali pengajar sudah menyiapkan folder aset.
